# Iris: modelo de escenas en Google Colab (profesor -> alumno)

Hace lo mismo que `scene/lambda/run_all.sh`, pero en Colab:

1. Baja fotos a color de Open Images.
2. Un modelo grande (el **profesor**, Qwen2.5-VL) las mira y escribe qué hay, con peligros: escalón, poste, hoyo, vehículo cerca...
3. Entrena el **alumno**, el modelo chico que corre en la placa XIAO.
4. Lo exporta a int8 para la placa.

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución* → **GPU A100** (la más rápida) o **L4**. La T4 **no sirve**: el profesor no cabe en su memoria.

**Si Colab se desconecta:** vuelve a correr todas las celdas desde arriba. Todo lo importante se guarda en tu Google Drive (carpeta `Iris_escenas`) y cada paso salta lo que ya estaba hecho.

**Al final**, baja de `Mi unidad/Iris_escenas/resultado/` los archivos que dice la última celda.

In [ ]:
#@title 1. Ajustes { display-mode: "form" }
FOTOS = 30000  #@param {type:"integer"}
PROFESOR = "Qwen/Qwen2.5-VL-7B-Instruct"  #@param ["Qwen/Qwen2.5-VL-7B-Instruct", "Qwen/Qwen2.5-VL-32B-Instruct"]
EPOCAS = 40  #@param {type:"integer"}
RAMA = "main"  #@param {type:"string"}
# 30.000 fotos con el profesor 7B: ~3-4 h en A100, ~7-9 h en L4 (estimado). Con 32B solo en A100 80 GB.

In [ ]:
#@title 2. Google Drive, código y GPU
import os, subprocess, pathlib
from google.colab import drive
drive.mount('/content/drive')

DRIVE = pathlib.Path('/content/drive/MyDrive/Iris_escenas')
DRIVE.mkdir(parents=True, exist_ok=True)
(DRIVE / 'resultado').mkdir(exist_ok=True)

REPO = pathlib.Path('/content/Proyecto_Iris')
if not REPO.exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-b', RAMA,
                    'https://github.com/yongiluu/Proyecto_Iris.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=False)
ML = REPO / 'ML_Training'
DATA = ML / 'data' / 'scene'
DATA.mkdir(parents=True, exist_ok=True)
os.chdir(ML)

gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip()
print('GPU:', gpu or 'NINGUNA')
assert gpu, 'No hay GPU: cambia el tipo de entorno de ejecución a A100 o L4.'
assert 'T4' not in gpu, 'La T4 no alcanza para el profesor: usa A100 o L4.'
MEM_GB = int(gpu.split(',')[-1].strip().split()[0]) / 1024

In [ ]:
#@title 3. Lo guardado en Drive vuelve a su lugar (si Colab se desconectó)
import json, tarfile, re

def enlazar(nombre):
    # El archivo vive en Drive; en el proyecto queda un acceso directo. Así cada línea escrita queda a salvo.
    en_drive, local = DRIVE / nombre, DATA / nombre
    en_drive.touch()
    if local.is_symlink() or local.exists():
        local.unlink()
    local.symlink_to(en_drive)

nombre_profesor = re.sub(r'[^a-z0-9.]+', '-', PROFESOR.split('/')[-1].lower()).strip('-')
ETIQ_AUTO = 'labels_auto.jsonl'
ETIQ_PROFESOR = f'labels_teacher_{nombre_profesor}.jsonl'
enlazar(ETIQ_AUTO)
enlazar(ETIQ_PROFESOR)

FOTOS_TAR = DRIVE / 'fotos.tar'
if FOTOS_TAR.exists() and not (DATA / 'images' / 'openimages').exists():
    print('Recuperando las fotos desde Drive...')
    with tarfile.open(FOTOS_TAR) as tar:
        tar.extractall(DATA)

# Si se cortó a mitad de la descarga, quita etiquetas de fotos que no alcanzaron a guardarse en Drive
# (la descarga las vuelve a bajar).
def limpiar(nombre):
    ruta = DRIVE / nombre
    filas = [l for l in ruta.read_text(encoding='utf-8').splitlines() if l.strip()]
    buenas = [l for l in filas if (DATA / 'images' / json.loads(l)['image']).exists()]
    if len(buenas) != len(filas):
        ruta.write_text(''.join(l + '\n' for l in buenas), encoding='utf-8')
    return len(buenas), len(filas) - len(buenas)

for n in (ETIQ_AUTO, ETIQ_PROFESOR):
    ok, quitadas = limpiar(n)
    print(f'{n}: {ok} filas' + (f' ({quitadas} sin foto, se rehacen)' if quitadas else ''))

In [ ]:
#@title 4. Fotos a color de Open Images
!pip -q install pandas pillow requests tqdm
!python scene/download_images.py --count {FOTOS} --workers 64

print('Guardando las fotos en Drive (una sola vez por tanda, tarda unos minutos)...')
tmp = pathlib.Path('/content/fotos.tar')
with tarfile.open(tmp, 'w') as tar:
    tar.add(DATA / 'images', arcname='images')
subprocess.run(['cp', str(tmp), str(FOTOS_TAR)], check=True)
tmp.unlink()
print('Fotos listas:', sum(1 for _ in (DATA / 'images' / 'openimages').glob('*.jpg')))

In [ ]:
#@title 5. Profesor (Qwen2.5-VL con vLLM): mira cada foto y escribe qué hay
# vLLM va en un entorno aparte para no romper el TensorFlow de Colab.
import time, urllib.request
VENV = pathlib.Path('/content/venv_profesor')
if not (VENV / 'bin' / 'vllm').exists():
    !pip -q install uv
    !uv venv -q {VENV} --python 3.12
    !uv pip install -q --python {VENV}/bin/python vllm requests tqdm pillow

# El muestreador de FlashInfer necesita kernels que Colab no trae (sin nvcc): vLLM se caia al arrancar.
os.environ['VLLM_USE_FLASHINFER_SAMPLER'] = '0'
largo = 8192 if MEM_GB >= 40 else 4096
log = open('/content/vllm.log', 'w')
servidor = subprocess.Popen([str(VENV / 'bin' / 'vllm'), 'serve', PROFESOR, '--port', '8000',
                             '--max-model-len', str(largo), '--gpu-memory-utilization', '0.92'],
                            stdout=log, stderr=subprocess.STDOUT)
print('Cargando el profesor (la primera vez baja ~16 GB, unos minutos)...')
while True:
    try:
        urllib.request.urlopen('http://localhost:8000/v1/models', timeout=5)
        break
    except Exception:
        if servidor.poll() is not None:
            lineas = open('/content/vllm.log').read().splitlines()
            errores = [l for l in lineas if 'Error' in l or 'ERROR' in l]
            print('\n'.join(errores[-25:] or lineas[-40:]))
            raise RuntimeError('vLLM se cayó: mira el log de arriba')
        time.sleep(10)
print('Profesor listo.')

PY = VENV / 'bin' / 'python'
print('-- Prueba con 10 fotos: revisa que las frases tengan sentido --')
!{PY} scene/label_teacher.py --url http://localhost:8000/v1 --model {PROFESOR} --limit 10
print('-- Todas las fotos (se puede cortar y retomar) --')
!{PY} scene/label_teacher.py --url http://localhost:8000/v1 --model {PROFESOR} --workers 64

servidor.terminate(); servidor.wait()
print('Profesor terminado; la GPU queda libre para el alumno.')

In [ ]:
#@title 6. Alumno: el modelo chico de la placa (TensorFlow de Colab con GPU)
!pip -q install pandas pillow tqdm
!python scene/train.py --labels mix --epochs {EPOCAS} --batch-size 128
!cp models/scene_student.keras models/scene_report.txt "{DRIVE}/resultado/" 2>/dev/null; true

In [ ]:
#@title 7. Exportar a int8 para la placa + demo, y guardar todo en Drive
!python scene/export_model.py
!python scene/demo.py --int8 --n 120
!cp models/scene_student_int8.tflite models/scene_report_int8.txt models/scene_student.keras models/scene_report.txt models/firmware_models.json "{DRIVE}/resultado/"
!cp data/scene/demo_alumno.html "{DRIVE}/resultado/" 2>/dev/null; true
!cp "{DRIVE}/{ETIQ_PROFESOR}" "{DRIVE}/resultado/"
print(open('models/scene_report_int8.txt', encoding='utf-8').read()[:3000])
print()
print('LISTO. Baja de Mi unidad/Iris_escenas/resultado/ a tu PC:')
print('  scene_student_int8.tflite, scene_report_int8.txt, scene_student.keras, scene_report.txt -> ML_Training/models/')
print(f'  {ETIQ_PROFESOR} -> ML_Training/data/scene/   (las etiquetas del profesor valen oro: guárdalas)')
print('  demo_alumno.html -> ábrelo en el navegador para ver qué dice el alumno de cada foto')
print('Y desconecta el entorno (Entorno de ejecución -> Desconectar y borrar) para no gastar unidades.')